# B v6 — replay kebijakan pasca-QC

Notebook penelitian: invalid ambigu/+0, proteksi origin–timeout–passage, kategori BUS dan panel audit. Tidak menjalankan detector, tracker, ReID, enhancement, atau tuning model. Bukan benchmark akurasi; input lama dan QC tetap tidak ditimpa. Baca `docs/B_V6_USER_POLICY_CONTRACT.md` dan review hasil aktual.


In [ ]:
from pathlib import Path
import sys
REPO_ROOT = Path('/kaggle/working/traffic-vehicle-counter-IDM-LOC')  # ubah lokasi checkout bila diperlukan
assert (REPO_ROOT / 'AGENTS.md').is_file(), 'Gunakan checkout repositori yang lengkap'
sys.path.insert(0, str(REPO_ROOT / 'notebooks'))
SOURCE_RUN_ROOT = Path('/kaggle/input/REPLACE_WITH_VERIFIED_DONOR/phase_8_tgdi_research/20261015T083731_664817Z')
SOURCE_VIDEO_ROOT = Path('/kaggle/input/datasets/chrisbiran/tgdi-traffic-videos')
OUTPUT_ROOT = REPO_ROOT / 'results/b_v6/execution'
RUN_ID = None  # None menghasilkan direktori baru dengan timestamp
VIDEO_IDS = ['weekday_siang']  # pilot; bukan delapan video
BRANCH = 'machine'  # review-overlay tetap terpisah


## Verifikasi perangkat lunak

Tes deterministik tidak menjalankan model. GPU tidak diperlukan untuk replay cache. Semua frame sumber diperlukan agar clock timeout tetap sah; jangan memotong atau menciptakan frame kosong.


In [ ]:
import unittest
suite = unittest.TestLoader().loadTestsFromNames([
    'phase_8_b_v6_identity_tests', 'phase_8_b_v6_counting_tests',
    'phase_8_b_v6_annotation_tests', 'phase_8_b_v6_batch_tests',
    'phase_9_v6_qc_review_tests'])
tests = unittest.TextTestRunner(verbosity=1).run(suite)
assert tests.wasSuccessful()


In [ ]:
import phase_8_b_v6_batch as batch
POLICY_PATH = REPO_ROOT / batch.POLICY_PATH
policy = batch.build_policy(REPO_ROOT)
if POLICY_PATH.exists():
    assert batch.old._read(POLICY_PATH) == policy, 'Konfigurasi berbeda; jangan menimpa profil tersimpan'
else:
    batch.old._write(POLICY_PATH, policy)
prepared = batch.prepare_b_v6_batch(repo_root=REPO_ROOT,
    source_run_root=SOURCE_RUN_ROOT, source_video_root=SOURCE_VIDEO_ROOT,
    video_ids=VIDEO_IDS, branch=BRANCH)
print({'jobs':len(prepared['jobs']), 'policy_sha256':prepared['policy_sha256'],
       'model_execution':False, 'timeout_source_frames':policy['timeout_frames']})


## Replay pilot penuh

Ini menghitung ulang aturan counting dari pengukuran model yang tersimpan. Tetap memakai raw preprocessing, geometri, threshold dan cache yang sama. Keputusan QC baru tidak diubah menjadi prediksi mesin.


In [ ]:
report = batch.run_b_v6_batch(repo_root=REPO_ROOT,
    source_run_root=SOURCE_RUN_ROOT, source_video_root=SOURCE_VIDEO_ROOT,
    video_ids=VIDEO_IDS, branch=BRANCH, output_root=OUTPUT_ROOT, run_id=RUN_ID)
print(report)
RUN_ROOT = Path(report['output_directory'])


## Ekspor dan anotasi

Pivot final accepted terpisah dari invalid/pending/not eligible; atribusi frame offline bukan waktu tersedianya keputusan. Video asli tidak disalin ke Git. `BUS=0` tidak membuktikan bus tidak ada. Rendering hanya membaca hasil replay.


In [ ]:
from phase_8_b_v6_annotation import export_audit, render_b_v6_video
for job in prepared['jobs']:
    slot = RUN_ROOT / job['video_id']
    result = batch.old._read(slot / 'counting_result.json.gz')
    geometry = batch.old._read(slot / 'geometry.json')
    exports = export_audit(result, slot / 'audit_exports')
    rendered = render_b_v6_video(job['video_path'], job['paths']['records'], result, geometry,
        slot / (job['video_path'].stem + '-b-v6-annotated.mp4'),
        fps=job['metadata']['fps'], expected_frame_count=job['metadata']['frame_count'],
        source_identity=result['source_identity'], unit_assignments=result['frame_assignments'],
        component_relations=result.get('component_links', []))
    print(exports, rendered)
# Segel ulang manifest hanya pada run baru setelah artifact ditambahkan.
batch.old._retention(RUN_ROOT)


## Review P0

Buka `weekday_siang/qc_review.json`, CSV audit dan video. Rekonsiliasi 68 pending +16 rekomendasi pembatalan +10 discovery historis; periksa4251/4281,4934/4953,4149/4367,4552/4480 dan passage balik. Hitung invalid sebagai eksklusi disengaja, bukan sebagai keberhasilan mendeteksi kendaraan. Pending turun bukan gate kelulusan. Periksa false count, duplikasi, salah gabung serta passage terlewat; belum ada ground truth lengkap atau bukti zero mistakes.
